# 06 — Reasoning-Oriented Prompting

## Scenario
A service is throwing 500 errors. We have the recent system logs. We need an AI to recommend a triage action. 

**The danger:** Naive prompts can jump to conclusions (such as "Restart the server") without checking evidence. We request concise, observable checks and verify the proposed action in application code; these artifacts are not private chain-of-thought or proof of correctness.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab06 import CASES, TriageRecommendation, VerificationResult, build_requests, decide, majority_vote, root_cause_named


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The Naive Direct Prompt

We ask for the answer directly. Models trained on internet text often default to "turn it off and on again" for technical issues if they aren't forced to read the context carefully.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i06/naive/pool-exhaustion")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
parsed = TriageRecommendation.model_validate_json(response.text)
print("PARSED:", parsed)
assert parsed.recommended_action == "restart the application service"


## Step 2: Structured, observable evidence checks

The schema requests concise log observations that a reviewer can verify. It does not request private chain-of-thought, and placing a field first does not guarantee correct reasoning. The application evaluates these observable checks and keeps execution authority outside the model.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i06/cot/pool-exhaustion")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
parsed = TriageRecommendation.model_validate_json(response.text)
print("PARSED:", parsed)
assert root_cause_named(parsed.evidence_checks, "connection pool exhausted")


## Step 3: Compound System (Planner / Verifier)

For critical actions (like actually executing code), a single LLM call is too risky. We split the reasoning into a Compound AI System: one call plans/proposes, a separate call (with a different prompt) verifies the proposal against safety rules.


In [ ]:
proposal = next(r for r in build_requests() if r.case_id == "i06/cot/pool-exhaustion")
proposal_response = client.generate(proposal)
proposal_value = TriageRecommendation.model_validate_json(proposal_response.text)
print("PLANNER RESPONSE:", proposal_response.text)
print("PLANNER PARSED:", proposal_value)
verification_request = next(r for r in build_requests() if r.case_id == "i06/verifier/pool-exhaustion")
show_request(verification_request)
response = client.generate(verification_request)
print("RECORDED RESPONSE:\n", response.text)
verification = VerificationResult.model_validate_json(response.text)
print("PARSED:", verification)
assert decide(proposal_value.recommended_action, verification) == "human_approval"


## Step 4: Self-consistency and rule-table control

Sampling several independent recommendations can expose disagreement, but application-side rules still decide whether an action may run.


In [ ]:
samples = []
for index in (1, 2, 3):
    request = next(r for r in build_requests() if r.case_id == f"i06/self-consistency/sample-{index}")
    show_request(request)
    response = client.generate(request)
    parsed = TriageRecommendation.model_validate_json(response.text)
    print("RECORDED RESPONSE:", response.text)
    print("PARSED:", parsed)
    samples.append(parsed.recommended_action)
print("VOTE:", majority_vote(samples))
assert majority_vote(samples) == ("Increase DB connection pool ceiling", 2 / 3)
optimistic_request = next(r for r in build_requests() if r.case_id == "i06/verifier/optimistic")
show_request(optimistic_request)
optimistic = client.generate(optimistic_request)
optimistic_value = VerificationResult.model_validate_json(optimistic.text)
print("PARSED:", optimistic_value)
assert decide("Restart DB_MAIN to clear the pool", optimistic_value) == "human_approval"


## Takeaway
The recorded pool incident improved root-cause naming while the verifier still routed the capacity-changing action to human approval; self-consistency selected the leading action in 2/3 samples.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Lab walkthrough](README.md#lab-walkthrough)
